In [1]:
import re
import os
import pandas as pd

In [2]:
# This script takes txt files from Elsevier website and processes them to get the data.
# This script was written utilising the initial processing script that was written by UROP student, Rachel Gupta.
def process_file(file_path):
    
    with open(file_path, "r", encoding="utf-8") as file:
        blocks = file.read().split('\n\n')

    def extract_field(regex_pattern, blocks, prefix='', group_num=1):
        pattern = re.compile(regex_pattern)
        field = [pattern.search(block).group(group_num) if pattern.search(block) else None for block in blocks]
        return field

    def extract_line(blocks, line_num):
        field = [block.split('\n')[line_num][:-1] if block.split('\n')[line_num] else None for block in blocks]
        return field

    extracted_data = [
        extract_line(blocks, 1),  # Title
        extract_line(blocks, 2),  # Journal Name
        [''] * len(blocks),  # Book Series Title
        extract_field(r'Volume (.*?),', blocks),  # Volume
        extract_field(r'Issue (.*?),', blocks),  # Issue
        extract_field(r'https://doi\.org/(\S+)\.', blocks, 'https://doi.org/'),  # DOI
        extract_field(r'Keywords: (.*?)$', blocks),  # Keywords
        extract_line(blocks, 0),  # Authors
        [int(year) if year else None for year in extract_field(r'(\d{4})', blocks)],  # Publication Year
        extract_field(r'\((.*?)\)', blocks),  # URL
        ['Article'] * len(blocks)  # Content Type
    ]

    return {col_name: data for col_name, data in zip(['Item Title', 'Publication Title', 'Book Series Title',
                                                      'Journal Volume', 'Journal Issue', 'Item DOI', 'Keywords', 'Authors',
                                                      'Publication Year', 'URL', 'Content Type'], extracted_data)}

In [ ]:
import os
import pandas as pd

base_directory = os.getcwd()
raw_data_path = os.path.join(base_directory, "raw_data")

output_directory = os.path.join(base_directory, "extracted_data")
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Dictionary to store DataFrames for each journal and country
dfs = {}

# List of journals (directories)
journals = ["applied_catalysis", "energy_policy", "greenhouse_gas_control", "hydrogen_energy",
           "solar_energy", "solar_energy_materials_and_solar_cells"]

# Process files for each journal and country
for journal in journals:
    journal_path = os.path.join(raw_data_path, journal)
    
    # Check if journal directory exists
    if not os.path.exists(journal_path):
        print(f"Journal directory does not exist: {journal_path}")
        continue
    
    # List all items in the journal directory
    try:
        all_items = os.listdir(journal_path)
        
        # Filter for only directories
        countries = [d for d in all_items if os.path.isdir(os.path.join(journal_path, d))]
        
        if not countries:
            print(f"No country directories found in {journal}")
            continue
            
        for country in countries:
            # Create key for storing DataFrame
            df_key = f"{journal}_{country}"
            dfs[df_key] = pd.DataFrame()
            
            # Path to country directory
            country_path = os.path.join(journal_path, country)
            
            # Process all txt files in country directory
            try:
                files = os.listdir(country_path)
                txt_files = [f for f in files if f.endswith('.txt')]
                
                for filename in txt_files:
                    file_path = os.path.join(country_path, filename)
                    try:
                        # Process file and add to appropriate DataFrame
                        extracted_data_dict = process_file(file_path)
                        df = pd.DataFrame(extracted_data_dict)
                        dfs[df_key] = pd.concat([dfs[df_key], df])
                    except Exception as e:
                        print(f"Error processing {file_path}: {str(e)}")

                if not dfs[df_key].empty:
                    csv_filename = f"{df_key}.csv"
                    csv_path = os.path.join(output_directory, csv_filename)
                    dfs[df_key].to_csv(csv_path, index=False)

            except Exception as e:
                print(f"Error accessing country directory {country_path}: {str(e)}")
                
    except Exception as e:
        print(f"Error accessing journal directory {journal_path}: {str(e)}")

print("\nProcessing Summary:")
print(f"Total number of DataFrames created: {len(dfs)}")
for key in dfs:
    print(f"{key}: {len(dfs[key])} rows")


Processing Summary:
Total number of DataFrames created: 48
applied_catalysis_canada: 163 rows
applied_catalysis_france: 391 rows
applied_catalysis_germany: 282 rows
applied_catalysis_singapore: 69 rows
applied_catalysis_south africa: 76 rows
applied_catalysis_switzerland: 46 rows
applied_catalysis_uk: 149 rows
applied_catalysis_us: 263 rows
energy_policy_canada: 163 rows
energy_policy_france: 391 rows
energy_policy_germany: 283 rows
energy_policy_singapore: 70 rows
energy_policy_south africa: 38 rows
energy_policy_switzerland: 46 rows
energy_policy_uk: 926 rows
energy_policy_us: 596 rows
greenhouse_gas_control_canada: 246 rows
greenhouse_gas_control_france: 160 rows
greenhouse_gas_control_germany: 210 rows
greenhouse_gas_control_singapore: 25 rows
greenhouse_gas_control_south africa: 19 rows
greenhouse_gas_control_switzerland: 57 rows
greenhouse_gas_control_uk: 308 rows
greenhouse_gas_control_us: 377 rows
hydrogen_energy_canada: 1505 rows
hydrogen_energy_france: 1425 rows
hydrogen_ene

In [13]:
print(dfs.keys())

dict_keys(['applied_catalysis_canada', 'applied_catalysis_france', 'applied_catalysis_germany', 'applied_catalysis_singapore', 'applied_catalysis_south africa', 'applied_catalysis_switzerland', 'applied_catalysis_uk', 'applied_catalysis_us', 'energy_policy_canada', 'energy_policy_france', 'energy_policy_germany', 'energy_policy_singapore', 'energy_policy_south africa', 'energy_policy_switzerland', 'energy_policy_uk', 'energy_policy_us', 'greenhouse_gas_control_canada', 'greenhouse_gas_control_france', 'greenhouse_gas_control_germany', 'greenhouse_gas_control_singapore', 'greenhouse_gas_control_south africa', 'greenhouse_gas_control_switzerland', 'greenhouse_gas_control_uk', 'greenhouse_gas_control_us', 'hydrogen_energy_canada', 'hydrogen_energy_france', 'hydrogen_energy_germany', 'hydrogen_energy_singapore', 'hydrogen_energy_south africa', 'hydrogen_energy_switzerland', 'hydrogen_energy_uk', 'hydrogen_energy_us', 'solar_energy_canada', 'solar_energy_france', 'solar_energy_germany', 'so

In [14]:
dfs['applied_catalysis_canada']

,Item Title,Publication Title,Book Series Title,Journal Volume,Journal Issue,Item DOI,Keywords,Authors,Publication Year,URL,Content Type
0,Improved carbon disulfide conversion: Modifica...,Applied Catalysis A: General,,604,None,10.1016/j.apcata.2020.117773,Claus catalyst; Alumina; Transition metal oxid...,"Ruohong Sui, Christopher B. Lavery, Connor E. ...",2020,https://www.sciencedirect.com/science/article/...,Article
1,Stability of Mo2C/carbon catalysts during dibe...,Applied Catalysis A: General,,600,None,10.1016/j.apcata.2020.117628,Molybdenum carbide; Hydrodeoxygenation (HDO); ...,"Shida Liu, Haiyan Wang, Chang Soo Kim, Kevin J...",2020,https://www.sciencedirect.com/science/article/...,Article
2,Bimetallic catalysts: Requirements for stabili...,Applied Catalysis A: General,,482,None,10.1016/j.apcata.2014.06.016,Core–shell; Stabilizer; Structure control; PVP...,"Hessam Ziaei-Azad, Natalia Semagina",2014,https://www.sciencedirect.com/science/article/...,Article
3,Impacts of vanadium and coke deposits on the C...,Applied Catalysis A: General,,504,None,10.1016/j.apcata.2014.11.006,Gasification; Nickel; Vanadium; Coke; Petroleu...,"Sip Chen Liew, Josephine M. Hill",2015,https://www.sciencedirect.com/science/article/...,Article
4,Synthesis and characterization of mesoporous a...,Applied Catalysis A: General,,489,None,10.1016/j.apcata.2014.10.008,Mesoporous aluminas; Hydrotreating; Hydrodesul...,"Sandeep Badoga, Rajesh V. Sharma, Ajay K. Dala...",2015,https://www.sciencedirect.com/science/article/...,Article
...,...,...,...,...,...,...,...,...,...,...,...
58,Catalyst design and tuning for oxidative dehyd...,Applied Catalysis A: General,,609,None,10.1016/j.apcata.2020.117914,Catalyst design; Oxidative dehydrogenation of ...,"Yahya Gambo, Sagir Adamu, Abdulrahman A. Abdul...",2021,https://www.sciencedirect.com/science/article/...,Article
59,Improving activity of ZSM-5 zeolite catalyst f...,Applied Catalysis A: General,,563,None,10.1016/j.apcata.2018.07.003,Catalytic fast pyrolysis; Hydrolysis lignin; A...,"Hooman Paysepar, Kasanneni Tirumala Venkateswa...",2018,Charles,Article
60,Liquid phase chemo-selective catalytic hydroge...,Applied Catalysis A: General,,454,None,10.1016/j.apcata.2012.12.010,Furfural; Furfuryl alcohol; Heterogeneous cata...,"Rajesh V. Sharma, Umashankar Das, Ramaswami Sa...",2013,https://www.sciencedirect.com/science/article/...,Article
61,CO2-mediated oxidative dehydrogenation of ligh...,Applied Catalysis A: General,,623,None,10.1016/j.apcata.2021.118273,Catalyst design; CO2 utilization; Olefins; Oxi...,"Yahya Gambo, Sagir Adamu, Gazali Tanimu, Ibrah...",2021,https://www.sciencedirect.com/science/article/...,Article
